In [ ]:
import pandas as pd
import yaml
 
try:
    with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

boardgames_df = pd.read_csv(config['input_data']['file'])

In [ ]:
boardgames_df

In [ ]:
boardgames_df = boardgames_df.drop(columns=["release_year", "rank_overall", "owned", "fans", "page_views", "std_deviation",	"comments", "monthly_plays", "previously_owned", "for_trade", "want_trade", "rating_1", "rating_2", "rating_3", "rating_4", "rating_5", "rating_6", "rating_7", "rating_8", "rating_9", "rating_10", "mechanics", "families", "designers", "artists", "publishers", "solo_designers", "developers", "graphic_designers", "sculptors", "editors", "writers", "insert_designers", "rank_strategy", "rank_thematic", "rank_family", "rank_war", "rank_customizable", "rank_abstract", "rank_party", "rank_childrens"])
boardgames_df = boardgames_df.rename(columns={"row_id": "game_id"})
boardgames_df["categories"] = boardgames_df["categories"].fillna("Unknown")
#boardgames_df = boardgames_df["categories"].replace("NaN", "Unknown")
boardgames_df

In [ ]:
boardgames_df['categories'].isna().sum()

In [ ]:
#unique categories list
categories_df = boardgames_df['categories'].str.split(";",expand=True)

display(categories_df.head())
#categories_df.info()
categories_list = []

for col in range(categories_df.shape[1]):
    for row in range(categories_df.shape[0]):
        if categories_df.iloc[row, col] != 'None' and categories_df.iloc[row, col] not in categories_list:
            categories_list.append(categories_df.iloc[row, col])

categories_list

In [ ]:
categories_names_df = pd.DataFrame({"category_id": [ i + 1 for i in range(len(categories_list))], "name": categories_list})
display(categories_names_df)
categories_names_df.to_csv("../data/clean/categories.csv", sep=";", encoding="utf-8", index=False)

In [ ]:
#csv file for playtime
playtime_df = boardgames_df[['min_playtime', 'max_playtime']].drop_duplicates()
playtime_df['playtime_id'] = [i + 1 for i in range(playtime_df.shape[0])]
playtime_df = playtime_df[['playtime_id', 'min_playtime', 'max_playtime']]
display(playtime_df)
playtime_df.to_csv("../data/clean/playtime.csv", sep=";", encoding="utf-8", index=False)

In [ ]:
#csv file for players
boardgames_df["minimum_age"] = boardgames_df["minimum_age"].replace(0, 4)
players_df = boardgames_df[['min_players', 'max_players','minimum_age']].drop_duplicates()
players_df['player_id'] = [i + 1 for i in range(players_df.shape[0])]
players_df = players_df[['player_id','min_players', 'max_players','minimum_age']]
display(players_df)
players_df.to_csv("../data/clean/players.csv", sep=";", encoding="utf-8", index=False)

In [ ]:
#csv file for games
merged_df = pd.merge(playtime_df, boardgames_df, left_on=['min_playtime','max_playtime'], right_on=['min_playtime','max_playtime'], how='right')

merged_2_df = pd.merge(players_df, merged_df, left_on=['min_players', 'max_players','minimum_age'], right_on=['min_players', 'max_players','minimum_age'], how='right')

games_df = merged_2_df.drop(['min_players', 'max_players', 'minimum_age', 'min_playtime', 'max_playtime'], axis=1)

games_df = games_df[['game_id', 'boardgame', 'avg_rating', 'num_ratings', 'complexity', 'wishlisted', 'total_plays', 'amazon_price', 'url', 'description', 'player_id', 'playtime_id']]
display(games_df.head())
games_df.to_csv("../data/clean/games.csv", sep=";", encoding="utf-8", index=False)

In [ ]:
#csv file for game_to_category
categories_names_df["name"] = categories_names_df["name"].fillna("Unknown")
categories_names_df


In [ ]:
for index, value in boardgames_df['categories'].items():
    continue
    #print(index, value)

In [ ]:
#[{"colname1": value1, "colname2":value2}]
cat = []

for row_category, category_name in categories_names_df['name'].items():
    for row_board in range(boardgames_df.shape[0]):
        if category_name in boardgames_df.iloc[row_board, list(boardgames_df.columns).index('categories')]:
            cat.append({"game_id": boardgames_df.iloc[row_board, 0], "category_id": categories_names_df.iloc[row_category, 0]})

game_to_category_df = pd.DataFrame(cat)

In [ ]:
game_to_category_df

In [ ]:
game_to_category_df.to_csv("../data/clean/game_to_category.csv", sep=";", encoding="utf-8", index=False)